## Приятные мелочи Питона

In [1]:
import pymorphy3
import re 
import math
import numpy as np

Давайте продолжим развивать наш учебный класс.<br>
Еще раз внимательно посмотрим на список методов нашего класса.

А что это за функции с двумя подчеркиваниями? И все ли из них такие private? <br>
На самом деле нет. Часть из этих функций - это синонимы для операторов. Мы ведь можем складывать два множества или матрицы. Каждый раз когда мы пишем<br>
a = b + c<br>
на самом деле вызывается следующий код.<br>
a = \__add\__(b, c)<br>
Операторы являются удобным представлением вызовов функций. Для того, чтобы определить соответствующий оператор надо просто добавить функцию в соответствующий класс. Список всех возможных операторов записан <a href="https://docs.python.org/3.7/library/operator.html">здесь</a> и <a href="https://docs.python.org/3/reference/datamodel.html#special-method-names">здесь</a>.<br>
То есть каждый класс может завести себе, например, оператор сложения, если ему это необходимо.<br>
Давайте немного преобразим наш класс. Пусть один объект можно будет сложить с другим, после чего у него пополнится словарь. А в наш класс можно будет отправить строку при помощи оператора <<, а оператор вернет векторное представление. И еще кое-что разной степени приятности.

In [87]:
# Вся перегрузка операторов находится внизу класса.
class FasterMorphology2:
    """ Класс для быстрого морфологического анализа текстов и их векторизации.
    """

    def __init__(self):  # Функция инициализации объекта после его создания.
        self.morpho = pymorphy2.MorphAnalyzer()
        self.cash = {}
        # Добавим словарь для запоминания, на каком месте вектора находится какая начальная форма.
        self.dictionary = {}

    def analyzeWords(self, words):
        """ Проводит морфологический анализ списка токенов words.
            Возвращает список начальных форм слов.
        """
        res = []
        for w in words:
            if w in self.cash:  # Сперва ищем очередное слово в кеше.
                res.append(self.cash[w])
            else:  # Если его там нет, проводим морфологический анализ и кешируем.
                r = self.morpho.parse(w)[0].normal_form
                res.append(r)
                self.cash[w] = r
                # Также для каждой начальной формы запоминаем ее позицию в векторе.
                if r not in self.dictionary:
                    self.dictionary[r] = len(self.dictionary)
        return res

    def breakByWords(self, text):
        """ Разбивает текст на русские слова.
        """
        return [w[0].lower() for w in re.findall("([А-ЯЁа-яё]+(-[А-ЯЁа-яё]+)*)", text)]

    def analyzeText(self, text):
        """ Проводит морфологический анализ строки с текстом text. 
            Выделяет из нее слова, написанные русской кириллицей.
            Возвращает список начальных форм слов.
        """
        words = self.breakByWords(text)
        return self.analyzeWords(words)

    # Вообще-то тоже самое умеет Counter, но ему надо сперва привести слова к начальной форме.
    def vectorizeAsDict(self, words):
        """ Возвращает векторное разреженное представление текста в виде словаря.
            Текст передается как список токенов words.
            Вместо позиции для индексации используется само слово.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words, str):
            words = self.breakByWords(words)

        vct = {}
        for word in words:  # Для каждого слова прповодим анализ.
            if word in self.cash:
                # Считаем частоты слов.
                vct[self.cash[word]] = vct.get(self.cash[word], 0)+1
            else:
                r = self.morpho.parse(word)[0].normal_form
                res.append(r)
                self.cash[word] = r
                vct[r] = vct.get(r, 0)+1
                if r not in self.dictionary:
                    self.dictionary[r] = len(self.dictionary)
        return vct

    def clearDict(self):
        """ Очищает словарь. Вдруг надо пересчитать так как изменилась размерность пространства.
        """
        self.dictionary = {}

    def formDict(self, texts):
        """ Сформировать словарь по тексту не формируя разметку текста.
        """
        for text in texts:
            for word in text:
                if word not in self.cash:
                    r = self.morpho.parse(word)[0].normal_form
                    self.cash[word] = r
                    if r not in self.dictionary:
                        self.dictionary[r] = len(self.dictionary)

    def vectorizeAsList(self, words2):
        """ Возвращает векторное представление текста в виде плотного списка (включает нули).
            Текст передается как список токенов words.
            Позиция каждого слова в векторе определяется числом, хранимым в dictionary.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words2, str):
            words = self.breakByWords(words2)
        else:
            words = words2

        # Сперва обновляем dictionary.
        for word in words:
            if word not in self.cash:
                r = self.morpho.parse(word)[0].normal_form
                self.cash[word] = r
                if r not in self.dictionary:
                    self.dictionary[r] = len(self.dictionary.keys())
        # Теперь, когда все слова есть в кеше и словаре и известен размер вектора, можно приступать к векторизации.
        vct = [0 for _ in self.dictionary]
        for word in words:
            vct[self.dictionary[self.cash[word]]] += 1
        return vct

    def vectorizeAsList2(self, words):
        """ Возвращает векторное представление текста в виде плотного списка (включает нули).
            Текст передается как список токенов words. В вектор включаются только слова, находящиес в словаре.
            Позиция каждого слова в векторе определяется числом, хранимым в dictionary.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        vct = [0 for _ in self.dictionary]
        for word in words:
            if word in self.cash:
                vct[self.dictionary[self.cash[word]]] += 1
        return vct

    def vectorizeAsArray(self, words):
        """ Возвращает векторное представление текста в виде плотного массива (включает нули).
            Текст передается как список токенов words.
            Позиция каждого слова в векторе определяется числом, хранимым в dictionary.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words, str):
            words = self.breakByWords(words)

        # Сперва обновляем dictionary.
        for word in words:
            if word not in self.cash:
                r = self.morpho.parse(w)[0].normal_form
                self.cash[word] = r
                if r not in self.dictionary:
                    self.dictionary[r] = len(self.dictionary.keys())
        # Теперь, когда все слова есть в кеше и словаре и известен размер вектора, можно приступать к векторизации.
        vct = np.zeros((len(self.dictionary)))
        for word in words:
            vct[self.dictionary[self.cash[word]]] += 1
        return vct

    # Здесь мы заложили проблему. Функция не умеет считать расстояние между p.array.
    def cosineSimilarity(self, a, b):
        """ Функция расчета косинусной меры сходства между двумя векторными представлениями текста.
            Работает по-разному в зависимости от представления вектора.
        """
        if type(a) != type(b):  # Тип векторов должен совпадать.
            return None
        # Если это списки, значит это плотное представление вектора.
        if isinstance(a, list):
            # Длины векторов в этом случае должны совпадать.
            if len(a) == 0 or len(b) == 0 or len(a) != len(b):
                return 0
            sumab = sum([a[na]*b[na] for na in range(len(a))])
            suma2 = sum([a[na]*a[na] for na in range(len(a))])
            sumb2 = sum([b[na]*b[na] for na in range(len(a))])
            return sumab/math.sqrt(suma2*sumb2)
        # Разреженное представление вектора - хранятся только ненулевые значения.
        elif isinstance(a, dict):
            # Вектора должны хранить хоть что-то.
            if len(a.keys()) == 0 or len(b.keys()) == 0:
                return 0
            sumab = sum([a[na] * b[na] for na in set(a.keys()) & set(b.keys())])
#            sumab=sum([a[na]*b[na] for na in a.keys() if na in b.keys()])
            suma2 = sum([a[na] * a[na] for na in a.keys()])
            sumb2 = sum([b[nb] * b[nb] for nb in b.keys()])
            return sumab / math.sqrt(suma2 * sumb2)
        return 0

    def JaccardCoefficient(self, a, b):
        """ Коэффициент Жаккара - отношение количества слов, встречающихся в обоих текстах к объединению лексики.
        """
        if type(a) != type(b):  # Тип векторов должен совпадать.
            return None
        # Если это списки, значит это плотное представление вектора.
        if isinstance(a, list):
            # Длины векторов в этом случае должны совпадать.
            if len(a) == 0 or len(b) == 0 or len(a) != len(b):
                return 0
            union = len(a) - [aa*bb for aa, bb in zip(a, b)].count(0)
            intersection = len(a)-[aa+bb for aa, bb in zip(a, b)].count(0)
            return union / intersection
        # Разреженное представление вектора - хранятся только ненулевые значения.
        elif isinstance(a, dict):
            # Вектора должны хранить хоть что-то.
            if len(a.keys()) == 0 or len(b.keys()) == 0:
                return 0
            return len(set(a.keys()) & set(b.keys())) / len(set(a.keys()) | set(b.keys()))
        return 0

    def __iadd__(self, other):
        """ Оператор добавления словаря от другого объекта.
        """
        self.cash.update(other.cash)
        for word in set(other.dictionary.keys()) - set(self.dictionary.keys()):
            self.dictionary[word] = len(self.dictionary)
        return self

    def __lshift__(self, text):
        """ Оператор возвращает векторное представление текста
        """
        return self.vectorizeAsList(text)

    def __repr__(self):
        """ Текстовое представление объекта.
        """
        return "Object of class FasterMorphology2 <" + str(id(self)) + \
            ">\nCash size: " + str(len(self.cash)) + \
            "\nDictionary size: " + str(len(self.dictionary))

    def __getitem__(self, key):
        """ Возвращает элемент словаря при помощи квадратных скобок.
        """
        if isinstance(key, slice):
            return list(self.dictionary.keys())[key.start: key.stop: key.step]
        else:
            return list(self.dictionary.keys())[key]

    def __bool__(self):
        """ Класс ведет себя как булевская переменная. Проверяет было ли что-нибудь закешировано.
        """
        return len(self.dictionary) != 0

    def __call__(self):
        """ Объект класса можно "вызвать" как функцию. Можно просто переопределить оператор "круглые скобки".
        """
        print("-=* Overall results for FasterMorphology2*=-\nCash size: " +
              str(len(self.cash)) + "\nDictionary size: " + str(len(self.dictionary)))


Итак, мы добавили некоторые операторы в наш класс. Теперь опробуем их на трех произведениях.

In [88]:
with open("data/war_and_peace.txt", encoding="utf8") as fil:
    textWP = fil.read()

with open("data/Kard_Orson__Igra_Jendera.fb2") as fil:
    textEnd = fil.read()
    
textMart=""
for i in range(2, 33):
    with open("data/veyr/index_split_0"+"{:0>2}".format(i)+".xhtml") as fil:
        textMart += fil.read()


Сперва сделаем по старой технологии, с вызовом функций.

In [89]:
fasterWP = FasterMorphology2()
fasterMart = FasterMorphology2()

NameError: name 'pymorphy2' is not defined

In [10]:
%time vctWP = fasterWP.vectorizeAsList(textWP)
%time vctMart = fasterMart.vectorizeAsList(textMart)

CPU times: user 10.9 s, sys: 90 ms, total: 11 s
Wall time: 11.2 s
CPU times: user 3.48 s, sys: 4.72 ms, total: 3.48 s
Wall time: 3.56 s


In [11]:
%%time
fasterEnd = FasterMorphology2()
vctEnd = fasterEnd.analyzeText(textEnd)

CPU times: user 3.22 s, sys: 30.7 ms, total: 3.26 s
Wall time: 3.27 s


А теперь попробуем выполнить тоже самое, но при помощи новых перегруженных операторов. 

In [12]:
%%time
fasterEnd = FasterMorphology2()
print(fasterEnd) # Выводим объект.
if not fasterEnd: # Проверяем есть ли что-то в словаре.
    fasterEnd += fasterWP # Пополняем словарь.
    fasterEnd += fasterMart
print(fasterEnd)
vctEnd = fasterEnd << textEnd # Векторизуем текст.
fasterEnd() # Вызываем метод от объекта.

Object of class FasterMorphology2 <138725537093600>
Cash size: 0
Dictionary size: 0
Object of class FasterMorphology2 <138725537093600>
Cash size: 59297
Dictionary size: 21760
-=* Overall results for FasterMorphology2*=-
Cash size: 65696
Dictionary size: 23651
CPU times: user 1.8 s, sys: 31.9 ms, total: 1.83 s
Wall time: 1.85 s


Обратите внимание, что не все функции являются собственно операторами. Некоторые из них - это специальные функции, вызываемые в специальных обстоятельствах.

Теперь попробуем разобраться с передачей параметров в функции.<br>
Помните функцию, которая выбирала "значимые части речи" с ее довольно спорным списком частей речи? Было бы здорово, если бы пользователь мог передавать туда свой список частей речи, но при это неспециалист мог бы просто согласиться со списком автора функции.<br>
Для этого существуют значения параметров по умолчанию, которые прописываются в объявлении функции.

In [4]:
# Обратите внимание на присвоение значения posList - именно так задается значение по умолчанию.
def getMeaningfullWords(morph, text:str, posList:list=('ADJF', 'NOUN', 'VERB')):
    words = []
    tokens = re.findall('[А-Яа-яЁё]+\-[А-Яа-яЁё]+|[А-Яа-яЁё]+', text)
    for t in tokens:
        pv = morph.parse(t)
        if pv[0].tag.POS in posList:
            words.append(pv[0].normal_form)
    return words

morph = pymorphy2.MorphAnalyzer()

Теперь мы можем просто вызвать функцию со своим списком частей речи.

In [13]:
getMeaningfullWords(morph, textMart[1000:1500], ['ADJF', 'NOUN', 'VERB', 'PREP'])

['месяц',
 'самый',
 'значительный',
 'в',
 'жизнь',
 'обернуться',
 'кошмар',
 'знать',
 'прочесть',
 'этот',
 'строка',
 'думать',
 'в',
 'конец',
 'конец',
 'мой',
 'запись',
 'найти',
 'мочь',
 'год',
 'через',
 'сто',
 'для',
 'отчёт',
 'на',
 'шестой',
 'сутки',
 'погибнуть',
 'наш',
 'команда',
 'счесть',
 'мёртвый',
 'мочь',
 'мочь',
 'в',
 'мой',
 'честь',
 'объявить',
 'день',
 'национальный',
 'траур',
 'на',
 'мой',
 'страница',
 'в',
 'википедия',
 'появиться',
 'запись',
 'марк',
 'уотня']

А можем согласиться на список авторов функции и не передавать ничего.

In [14]:
getMeaningfullWords(morph, textMart[1000:1500])

['месяц',
 'самый',
 'значительный',
 'жизнь',
 'обернуться',
 'кошмар',
 'знать',
 'прочесть',
 'этот',
 'строка',
 'думать',
 'конец',
 'конец',
 'мой',
 'запись',
 'найти',
 'мочь',
 'год',
 'сто',
 'отчёт',
 'шестой',
 'сутки',
 'погибнуть',
 'наш',
 'команда',
 'счесть',
 'мёртвый',
 'мочь',
 'мочь',
 'мой',
 'честь',
 'объявить',
 'день',
 'национальный',
 'траур',
 'мой',
 'страница',
 'википедия',
 'появиться',
 'запись',
 'марк',
 'уотня']

Передача параметров может быть чрезвычайно удобна в нескольких случаях.
- В подавляющем большинстве случаев функция вызывается с значениями параметров по умолчанию. 
- Вам необходимо переделать дизайн функции так, чтобы в ней появились новые параметры, но при этом необходимо чтобы весь остальной код продолжал выполняться.
- Необходимо застраховаться от ошибки.
- Параметр по умолчанию принимает особое значение, которое нам о чем-то говорит.

Можно сделать несколько параметров со значениями по умолчанию. Если при этом не передавать последовательно несколько значений с конца, то можно просто их не писать. Если мы пропустили значение из середины списка, для остальных придется писать какому параметру какое значение передается.

In [90]:
def dummie1(a=0, b=1, c=2, d=3):
    """Annotation"""
    return a + b * c - d - 1

In [26]:
print(dummie1())
print(dummie1(2))
print(dummie1(2, 3))
print(dummie1(2, d=4))
print(dummie1.__defaults__)
print(dummie1.__code__)

-2
0
4
-1
(0, 1, 2, 3)
<code object dummie1 at 0x7e2b48a0f5d0, file "/tmp/ipykernel_6706/706599154.py", line 1>


In [91]:
dir(dummie1)

['__annotations__',
 '__builtins__',
 '__call__',
 '__class__',
 '__closure__',
 '__code__',
 '__defaults__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__globals__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__kwdefaults__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__type_params__']

In [83]:
dummie1.__annotations__, dummie1.__doc__

({}, 'Annotation')

In [84]:
dummie1.__dict__

{}

In [22]:
dir(dummie1.__code__), dummie1.__code__.co_varnames

(['__class__',
  '__delattr__',
  '__dir__',
  '__doc__',
  '__eq__',
  '__format__',
  '__ge__',
  '__getattribute__',
  '__gt__',
  '__hash__',
  '__init__',
  '__init_subclass__',
  '__le__',
  '__lt__',
  '__ne__',
  '__new__',
  '__reduce__',
  '__reduce_ex__',
  '__repr__',
  '__setattr__',
  '__sizeof__',
  '__str__',
  '__subclasshook__',
  'co_argcount',
  'co_cellvars',
  'co_code',
  'co_consts',
  'co_filename',
  'co_firstlineno',
  'co_flags',
  'co_freevars',
  'co_kwonlyargcount',
  'co_lines',
  'co_linetable',
  'co_lnotab',
  'co_name',
  'co_names',
  'co_nlocals',
  'co_posonlyargcount',
  'co_stacksize',
  'co_varnames',
  'replace'],
 ('a', 'b', 'c', 'd'))

In [29]:
dummie1.__code__.co_name

'dummie1'

In [23]:
for line in dummie1.__code__.co_lines():
    print(line)

(0, 16, 3)


В книге Л. Рамальо "Python: к вершинам мастерства" есть раздел с названием "Значения по умолчанию изменяемого типа: плохая идея" (с. 259). В ней приведен следующий пример. Пусть у нас есть класс автобуса, который хранит имена пассажиров.

In [30]:
class HauntedBus:
    """A bus model haunted by ghost passengers"""

    def __init__(self, passengers=[]):  # Вот здесь сделана ошибка, от которой потом все беды.
        self.passengers = passengers  

#     def __init__(self, passengers=None):  # Версия без ошибок.
#         if passengers:
#             self.passengers = passengers  
#         else:
#             self.passengers = []

    def pick(self, name):
        self.passengers.append(name)  

    def drop(self, name):
        self.passengers.remove(name)

        
bus1 = HauntedBus(['Alice', 'Bill'])
print(bus1.passengers)
#['Alice', 'Bill']
bus1.pick('Charlie')
bus1.drop('Alice')
print(bus1.passengers)
#['Bill', 'Charlie']
bus2 = HauntedBus()
bus2.pick('Carrie')
print(bus2.passengers)
#['Carrie']
bus3 = HauntedBus()
print(bus3.passengers)
#['Carrie']
bus3.pick('Dave')
print(bus2.passengers)
#['Carrie', 'Dave']
print(bus2.passengers is bus3.passengers)
#True
print(bus1.passengers)
#['Bill', 'Charlie']
print(dir(HauntedBus.__init__))
#['__annotations__', '__call__', ..., '__defaults__', ...]
print(HauntedBus.__init__.__defaults__) # Список значений по умолчанию для конструктора класса.
#(['Carrie', 'Dave'],)
print(HauntedBus.__init__.__defaults__[0] is bus2.passengers) # Оказывается всем спискам присвоена ссылка на объект по умолчанию!
#True
bus4 = HauntedBus()
print(bus4.passengers)
# ['Carrie', 'Dave']

['Alice', 'Bill']
['Bill', 'Charlie']
['Carrie']
['Carrie']
['Carrie', 'Dave']
True
['Bill', 'Charlie']
['__annotations__', '__builtins__', '__call__', '__class__', '__closure__', '__code__', '__defaults__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__get__', '__getattribute__', '__globals__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__kwdefaults__', '__le__', '__lt__', '__module__', '__name__', '__ne__', '__new__', '__qualname__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__']
(['Carrie', 'Dave'],)
True
['Carrie', 'Dave']


Разбирая этот пример мы можем увидеть, что если в конструкторе список пассажиров берется как значение по умолчанию, то списку пассажиров данного объекта присваивается ссылка на список по умолчанию (который сам по себе тоже переменная, точнее, ее значение). Соответственно, в дальнейшем все автобусы, которым мы не передали список, будут указывать на один и тот же список. <br>
Будьте бдительны! Делайте глубокую копию со значений по умолчанию изменяемого типа.

<h2>Произвольный список параметров функции</h2>

Теперь давайте разберемся как передать в функцию произвольное количество параметров.<br>
В Python можно передать переменное количество аргументов двумя способами:
- кортеж `*args` для позиционных аргументов;
- словарь `**kwargs` для именованных аргументов.

Мы используем `*args` и `**kwargs` в качестве аргумента, когда заранее не известно, сколько значений мы хотим передать функции. Функция может принимать оба этих параметра на случай, если передача будет вестись как с именем, так и без.

In [36]:
def mult(*multipliers):
    print(f'parameters: {multipliers}')
    res = 1
    for arg in multipliers:
        res *= arg
    return res

def HTMLizer(tag='n', *text, **args):
    print("positional:", text)
    print("named:", args)
    ar = ' '.join([a[0] + '="' + str(a[1]) + '"' for a in args.items()])
    txt = ' '.join([str(t) for t in text])
    return "<" + tag + " " + ar + ">" + txt + "</" + tag + ">"

In [40]:
print(mult(2, 3, 4, 5))
print(HTMLizer("p", "that is", "some text", size=12, color="red"))
print(HTMLizer("p"))

parameters: (2, 3, 4, 5)
120
positional: ('that is', 'some text')
named: {'size': 12, 'color': 'red'}
<p size="12" color="red">that is some text</p>
positional: ()
named: {}
<p ></p>


In [42]:
# No way!
print(mult(multipliers=(2, 3, 4, 5)))


TypeError: mult() got an unexpected keyword argument 'multipliers'

Обратите внимание, что при передаче параметра знак звездочки означает распаковку параметров. Например, функция принимает несколько параметров. Вместо этого Вы передаете кортеж с тем же числом полей, добавив к нему знак звездочки. 

In [90]:
asd = (2, 3, 4, 5)
asd2 = (2, 3, 4, 5)
mult(*asd, *asd2)

parameters: (2, 3, 4, 5, 2, 3, 4, 5)


14400

In [1]:
def dummie2(a, b):
    print(10*a+b)

ddd = ('1', '2')
dummie2(*ddd)

fff = {'b': 2, 'a': 5, 'c':10}
dummie2(**fff)# dummie2(a=5, b=2)
print(tuple(fff))

11111111112


TypeError: dummie2() got an unexpected keyword argument 'c'

Это работает не только как в примере с произвольными параметрами, но и на любых других функциях.

In [92]:
def printCoords(long, lat):
    print("Longitude: ", long, ", latitude: ", lat)
    
coords = [(1.23, 2.34), (3.45, 6.45), (6.76, 8.98)]
for coord in coords:
    printCoords(*coord)

Longitude:  1.23 , latitude:  2.34
Longitude:  3.45 , latitude:  6.45
Longitude:  6.76 , latitude:  8.98


Также можно принять в функцию список аргументов по имени и в зависимости от того, что в нем есть производить те или иные действия.

In [93]:
def printCoords(long, lat, **kwargs):
    res = ""
    if 'planet' in kwargs.keys():
        res += 'Coordinates on planet ' + kwargs['planet'] + " are "
    res += "Longitude: " + str(long) + ", Latitude: " + str(lat)
    print(res)
    
coords=[(1.23, 2.34), (3.45, 6.45), (6.76, 8.98)]
planets=["Earth", "Mars", "Krypton"]
for coord in coords:
    printCoords(*coord)
for coord, planet in zip(coords, planets):
    printCoords(*coord, planet=planet, strange='high')

Longitude: 1.23, Latitude: 2.34
Longitude: 3.45, Latitude: 6.45
Longitude: 6.76, Latitude: 8.98
Coordinates on planet Earth are Longitude: 1.23, Latitude: 2.34
Coordinates on planet Mars are Longitude: 3.45, Latitude: 6.45
Coordinates on planet Krypton are Longitude: 6.76, Latitude: 8.98


Начиная с Python 3.8 можно явно задать какие параметры должны передаваться только как позиционные, а какие как именованные. Для этого используются знаки / и * в списке параметров. Те параметры, которые находятся до / , должны передаваться только как позиционные. Те параметры, которые находятся после * , должны передаваться как именованные. Параметры между этими значками могут передаваться как угодно.

In [31]:
def parameter_demonstrator1(a, b, /, c, d, *, e, f):
    pass

In [32]:
# Все нормально: позиционные в начале, именованные переданы по имени, остальные - позиционно.
parameter_demonstrator1(1, 2, 3, 4, e=5, f=6)

In [33]:
# Все нормально: позиционные в начале, именованные переданы по имени, остальные - по имени.
parameter_demonstrator1(1, 2, d=3, c=4, e=5, f=6)

In [34]:
# Ошибка - позиционные параметры переданы по имени.
parameter_demonstrator1(a=1, b=2, 3, 4, e=5, f=6)

SyntaxError: positional argument follows keyword argument (573361232.py, line 2)

In [98]:
# Ошибка - именованные параметры переданы позиционно.
parameter_demonstrator1(1, 2, 3, 4, 5, 6)

TypeError: parameter_demonstrator1() takes 4 positional arguments but 6 were given

Можно создавать функции, все параметры которых будут только позиционными, только именованными или не будут разрешать смешанный стиль

In [99]:
def parameter_demonstrator2(*, e, f):
    pass

In [100]:
parameter_demonstrator2(f=1, e=2)

In [101]:
parameter_demonstrator2(1, 2)

TypeError: parameter_demonstrator2() takes 0 positional arguments but 2 were given

В сочетании со значениями по умолчанию можно получить мощный инструмент, заставляющий документировать вызов функции, явно показывая какие значения какие параметры принимают. Или заставляющий пользователя функции оформлять параметры в виде словарей.

In [14]:
def parameter_demonstrator3(*, a=1, b=2):
    pass

In [103]:
parameter_demonstrator3()

In [104]:
parameter_demonstrator3(b=10)

In [15]:
old_parameter_demonstrator3 = parameter_demonstrator3
def my_func(a=1, b=2):
    print(1)
    old_parameter_demonstrator3(a=a, b=b)
    print(2)

parameter_demonstrator3 = my_func
    
parameter_demonstrator3(a=1, b=2)

1
2


<h2>Декораторы функций</h2>

Теперь перейдем к декораторам. (Качественное дополнительное чтение [здесь](https://habr.com/ru/articles/710654/).)

Иногда нам необходимо расширить возможности какой-то функции. Для этих целей в Питоне используются декораторы, то есть функции, которые принимают другую функцию и возвращают третью (в общем случае).

In [16]:
def trace(func):
    def inner(*args, **kwargs):
        print("calling function:", func.__name__, ", parameters", args, kwargs)
        return func(*args, **kwargs)
    return inner

@trace
def calc(a, b):
    return a + b

# calc = trace(calc)

calc(1, 2)

calling function: calc , parameters (1, 2) {}


3

In [17]:
#  Аналогично
def trace(func):
    def inner(*args, **kwargs):
        print("callingg function:", func.__name__, ", parameters", 
              args, kwargs)
        return func(*args, **kwargs)
    return inner

def calc(a, b):
    return a + b

calc = trace(calc)
calc(1, 2)
# inner(1, 2)

callingg function: calc , parameters (1, 2) {}


3

Обратите внимание, что функций в общем случае три:
- декорируемая;
- декорирующая;
- та, которой декорируют.

Общая идея декоратора состоит в том, что мы можем некоторым образом попросить вторую функцию сделать так, чтобы вместо первой вызывалась третья. При этом третья знает о существовании первой и может использует результаты ее работы.<br>
В некотором роде, мы не заявляем, что декорируем первой функцией. Мы скромно просим задекорировать для нас первую функцию.

Задекорировать функцию можно несколькими другими функциями. Порядок декораторов имеет значение.

In [18]:
from math import sin, cos

def operations(func, x):
    return func(x) * func(x)

# if feature1:
#     f2 = lambda x: (x + 1) * (x + 1)
# elif feature2:
#     f2 = sin
# else:
#     f2 = cos
    
# f2(asd)

operations(sin, 1), operations(cos, 1)
# operations(f2, sin, 2)

(0.7080734182735712, 0.2919265817264289)

In [19]:
f2(sin, 3)

NameError: name 'f2' is not defined

In [108]:
# # asd is a function which returns square of ...
# #     x - function parameter.
# def asd(x):
#     """ kjdjskdfghsfdghshbsjkdf hgjhgsdjk ghsdjkghsjkdfsghfdjkn
#         dfkjkvnsdfjghdfsjfjk 
#          adff,gjhsdfjkghdjk
#     """
#     return x*x

whom2call = {"asd": lambda x: x*x,
             "qwe": lambda x: x+x,
             "zxc": lambda x: x/x,
             "sin": sin,
             "cos": cos
            }

def some_strange(**kwargs):
    for func, val in kwargs.items():
        print(whom2call[func](val))
        
some_strange(**{"asd":1, "zxc":3, "sin":4})        
# some_strange(asd=1, zxc=3, sin=4)        

1
1.0
-0.7568024953079282


In [20]:
# https://compscicenter.ru/media/slides/python_2015_autumn/2015_09_21_python_2015_autumn_93c2yAw.pdf

def square(func):
    return lambda x: func(x * x)

def addsome(func):
    return lambda x: func(x + 42)

@square
@addsome
def identity(x):
    return x

print(identity(2))
# 46
@addsome
@square
def identity(x):
    return x
print(identity(2))
# 1936

46
1936


Важное замечание про декораторы. <br>
Декорирующие функции выполняются по мере объявления.

In [21]:
# https://compscicenter.ru/media/slides/python_2015_autumn/2015_09_21_python_2015_autumn_93c2yAw.pdf
cntr_sq = 0

def square(func):
    global cntr_sq
    cntr_sq += 1
    print(f"square {cntr_sq}")
    return lambda x: func(x * x)

def addsome(func):
    print("addsome")
    return lambda x: func(x + 42)

@square
@addsome
def identity1(x):
    print("identity 1")
    return x
print("-----")

@addsome
@square
def identity2(x):
    print("identity 2")
    return x

addsome
square 1
-----
square 2
addsome


In [23]:
print(identity1(2))
# 46

print(identity2(2))
# 1936

identity 1
46
identity 2
1936


Получается, что место, где декорируется функция, например, вот такое.

`
@first
def second():
    pass
`

Так вот в реальности оно выглядит вот так.

`
def second():
    pass
second=first(second)
`


Теперь посмотрим на библиотеку с декораторами `functools`. Например, `functools.lru_cache` кеширует результаты и при повторном вызове с теми же параметрами подставляет результаты из кеша.

In [24]:
import functools

In [25]:
@trace
def fibonacci(n): # Посчитаем числа Фибоначи.
    if n < 2:
        return n
    return fibonacci(n-2) + fibonacci(n-1)

@trace
@functools.lru_cache()
def fibonacci2(n):
    print(f"actually calc {n}")
    if n<2:
        return n
    return fibonacci2(n-2) + fibonacci2(n-1)

@functools.lru_cache()
@trace
def fibonacci3(n):
    if n<2:
        return n
    return fibonacci3(n-2) + fibonacci3(n-1)


In [26]:
print(fibonacci(6))
print("-----")
print(fibonacci2(6))
print("-----")
print(fibonacci3(6))

callingg function: fibonacci , parameters (6,) {}
callingg function: fibonacci , parameters (4,) {}
callingg function: fibonacci , parameters (2,) {}
callingg function: fibonacci , parameters (0,) {}
callingg function: fibonacci , parameters (1,) {}
callingg function: fibonacci , parameters (3,) {}
callingg function: fibonacci , parameters (1,) {}
callingg function: fibonacci , parameters (2,) {}
callingg function: fibonacci , parameters (0,) {}
callingg function: fibonacci , parameters (1,) {}
callingg function: fibonacci , parameters (5,) {}
callingg function: fibonacci , parameters (3,) {}
callingg function: fibonacci , parameters (1,) {}
callingg function: fibonacci , parameters (2,) {}
callingg function: fibonacci , parameters (0,) {}
callingg function: fibonacci , parameters (1,) {}
callingg function: fibonacci , parameters (4,) {}
callingg function: fibonacci , parameters (2,) {}
callingg function: fibonacci , parameters (0,) {}
callingg function: fibonacci , parameters (1,) {}


Теперь попробуем аккуратно заделать недостаток нашего класса, который состоит в том, что мы не умеем считать косинусную меру для `numpy.array`. Будем использовать для этого `functools.singledispatch`.

In [27]:
# Все декораторы вынесены из определения класса, так как не дело это класса считать косинусную меру. Она сама по себе.
# А еще я заменил действие оператора << , теперь он возвращает словарь. Так все-таки проще считать косинусную меру.
class FasterMorphology2:
    """ Класс для быстрого морфологического анализа текстов и их векторизации.
    """

    def __init__(self):  # Функция инициализации объекта после его создания.
        self.morpho = pymorphy2.MorphAnalyzer()
        self.cash = {}
        # Добавим словарь для запоминания, на каком месте вектора находится какая начальная форма.
        self.dictionary = {}

    def analyzeWords(self, words):
        """ Проводит морфологический анализ списка токенов words.
            Возвращает список начальных форм слов.
        """
        res = []
        for w in words:
            if w in self.cash:  # Сперва ищем очередное слово в кеше.
                res.append(self.cash[w])
            else:  # Если его там нет, проводим морфологический анализ и кешируем.
                r = self.morpho.parse(w)[0].normal_form
                res.append(r)
                self.cash[w] = r
                # Также для каждой начальной формы запоминаем ее позицию в векторе.
                if r not in self.dictionary:
                    self.dictionary[r] = len(self.dictionary)
        return res

    def breakByWords(self, text):
        """ Разбивает текст на русские слова.
        """
        return [w[0].lower() for w in re.findall("([А-ЯЁа-яё]+(-[А-ЯЁа-яё]+)*)", text)]

    def analyzeText(self, text):
        """ Проводит морфологический анализ строки с текстом text. 
            Выделяет из нее слова, написанные русской кириллицей.
            Возвращает список начальных форм слов.
        """
        words = self.breakByWords(text)
        return self.analyzeWords(words)

    # Вообще-то тоже самое умеет Counter, но ему надо сперва привести слова к начальной форме.
    def vectorizeAsDict(self, words):
        """ Возвращает векторное разреженное представление текста в виде словаря.
            Текст передается как список токенов words.
            Вместо позиции для индексации используется само слово.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words, str):
            words = self.breakByWords(words)

        vct = {}
        for word in words:  # Для каждого слова прповодим анализ.
            if word in self.cash:
                # Считаем частоты слов.
                vct[self.cash[word]] = vct.get(self.cash[word], 0) + 1
            else:
                r = self.morpho.parse(word)[0].normal_form
                self.cash[word] = r
                vct[r] = vct.get(r, 0) + 1
                if r not in self.dictionary:
                    self.dictionary[r] = len(self.dictionary)
        return vct

    def clearDict(self):
        """ Очищает словарь. Вдруг надо пересчитать так как изменилась размерность пространства.
        """
        self.dictionary = {}

    def formDict(self, texts):
        """ Сформировать словарь по тексту не формируя разметку текста.
        """
        for text in texts:
            for word in text:
                if word not in self.cash:
                    r = self.morpho.parse(word)[0].normal_form
                    self.cash[word] = r
                    if r not in self.dictionary:
                        self.dictionary[r] = len(self.dictionary)

    def vectorizeAsList(self, words2):
        """ Возвращает векторное представление текста в виде плотного списка (включает нули).
            Текст передается как список токенов words.
            Позиция каждого слова в векторе определяется числом, хранимым в dictionary.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words2, str):
            words = self.breakByWords(words2)
        else:
            words = words2

        # Сперва обновляем dictionary.
        for word in words:
            if word not in self.cash:
                r = self.morpho.parse(word)[0].normal_form
                self.cash[word] = r
                if r not in self.dictionary:
                    self.dictionary[r] = len(self.dictionary.keys())
        # Теперь, когда все слова есть в кеше и словаре и известен размер вектора, можно приступать к векторизации.
        vct = [0 for _ in self.dictionary]
        for word in words:
            vct[self.dictionary[self.cash[word]]] += 1
        return vct

    def vectorizeAsList2(self, words):
        """ Возвращает векторное представление текста в виде плотного списка (включает нули).
            Текст передается как список токенов words. В вектор включаются только слова, находящиес в словаре.
            Позиция каждого слова в векторе определяется числом, хранимым в dictionary.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        vct = [0 for _ in self.dictionary]
        for word in words:
            if word in self.cash:
                vct[self.dictionary[self.cash[word]]] += 1
        return vct

    def vectorizeAsArray(self, words):
        """ Возвращает векторное представление текста в виде плотного массива (включает нули).
            Текст передается как список токенов words.
            Позиция каждого слова в векторе определяется числом, хранимым в dictionary.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words, str):
            words = self.breakByWords(words)

        # Сперва обновляем dictionary.
        for word in words:
            if word not in self.cash:
                r = self.morpho.parse(w)[0].normal_form
                self.cash[word] = r
                if r not in self.dictionary:
                    self.dictionary[r] = len(self.dictionary.keys())
        # Теперь, когда все слова есть в кеше и словаре и известен размер вектора, можно приступать к векторизации.
        vct = np.zeros((len(self.dictionary)))
        for word in words:
            vct[self.dictionary[self.cash[word]]] += 1
        return vct

    def __iadd__(self, other):
        """ Оператор добавления словаря от другого объекта.
        """
        self.cash.update(other.cash)
        for word in set(other.dictionary.keys()) - set(self.dictionary.keys()):
            self.dictionary[word] = len(self.dictionary)
        return self

    def __lshift__(self, text):
        """ Оператор возвращает векторное представление текста в виде словаря.
        """
        return self.vectorizeAsDict(text)

    def __repr__(self):
        """ Текстовое представление объекта.
        """
        return "Object of class FasterMorphology2 <" + str(id(self)) + \
            ">\nCash size: " + str(len(self.cash)) + \
            "\nDictionary size: " + str(len(self.dictionary))

    def __getitem__(self, key):
        """ Возвращает элемент словаря при помощи квадратных скобок.
        """
        if isinstance(key, slice):
            return list(self.dictionary.keys())[key.start: key.stop: key.step]
        else:
            return list(self.dictionary.keys())[key]

    def __bool__(self):
        """ Класс ведет себя как булевская переменная. Проверяет было ли что-нибудь закешировано.
        """
        return len(self.dictionary) != 0

    def __call__(self):
        """ Объект класса можно "вызвать" как функцию. Можно просто переопределить оператор "круглые скобки".
        """
        print("-=* Overall results for FasterMorphology2*=-\nCash size: " +
              str(len(self.cash)) + "\nDictionary size: " + str(len(self.dictionary)))
        return self


In [28]:
@functools.singledispatch
# Говорим, что у нас есть функция, которую мы будем расширять.
def cosineSimilarity(a, b):
    return 0


# Навешиваем на нее декоратор, который будет вызываться только если первый параметр имеет тип list.
@cosineSimilarity.register(list)
def _(a, b):
    print("list function")
    # Длины векторов в этом случае должны совпадать.
    if len(a) == 0 or len(b) == 0 or len(a) != len(b):
        return 0
    sumab = sum([na * nb for na, nb in zip(a, b)])
    suma2 = sum([na * na for na in a])
    sumb2 = sum([nb * nb for nb in b])
    return sumab / math.sqrt(suma2 * sumb2)


@cosineSimilarity.register(dict)  # Еще один декоратор для типа dict.
def _(a, b):
    print("dict function")
    # Вектора должны хранить хоть что-то.
    if len(a.keys()) == 0 or len(b.keys()) == 0:
        return 0
    sumab = sum([a[na] * b[na] for na in set(a.keys()) & set(b.keys())])
    suma2 = sum([a[na] * a[na] for na in a.keys()])
    sumb2 = sum([b[nb] * b[nb] for nb in b.keys()])
    return sumab / math.sqrt(suma2 * sumb2)


@cosineSimilarity.register(np.ndarray)  # И декоратор для типа np.array.
def _(a, b):
    print("array function")
# Длины векторов в этом случае должны совпадать.
    if len(a) == 0 or len(b) == 0 or len(a) != len(b):
        return 0
    sumab = sum([na * nb for na, nb in zip(a, b)])
    suma2 = sum([na * na for na in a])
    sumb2 = sum([nb * nb for nb in b])
    return sumab / math.sqrt(suma2 * sumb2)


@functools.singledispatch
def JaccardCoefficient(a, b):  # Повторяем для коэффициента Жаккара.
    return 0


@JaccardCoefficient.register(list)
def _(a, b):
    # Длины векторов в этом случае должны совпадать.
    if len(a) == 0 or len(b) == 0 or len(a) != len(b):
        return 0
    union = len(a) - [aa * bb for aa, bb in zip(a, b)].count(0)
    intersection = len(a) - [aa + bb for aa, bb in zip(a, b)].count(0)
    return union/intersection


@JaccardCoefficient.register(dict)
def _(a, b):
    # Вектора должны хранить хоть что-то.
    if len(a.keys()) == 0 or len(b.keys()) == 0:
        return 0
    return len(set(a.keys()) & set(b.keys())) / len(set(a.keys()) | set(b.keys()))


@JaccardCoefficient.register(np.ndarray)
def _(a, b):
    # Длины векторов в этом случае должны совпадать.
    if len(a) == 0 or len(b) == 0 or len(a) != len(b):
        return 0
    union = len(a) - [aa * bb for aa, bb in zip(a, b)].count(0)
    intersection = len(a) - [aa + bb for aa, bb in zip(a, b)].count(0)
    return union / intersection


NameError: name 'np' is not defined

In [117]:
# Посмотрим как ведет себя косинус для разных типов.
a = [1, 2 ,3]
b = [3, 2, 1]

print(cosineSimilarity(a, b))

a = {1: 1, 2: 2, 3: 3}
b = {1: 3, 2: 2, 3: 1}

print(cosineSimilarity(a, b))
   
a = np.array(([1, 2, 3]))
b = np.array(([3, 2, 1]))

print(cosineSimilarity(a, b))

list function
0.7142857142857143
dict function
0.7142857142857143
array function
0.7142857142857143


А теперь посмотрим как будут вести себя векторы от нашей кешированной морфологии.

In [118]:
fasterWP = FasterMorphology2()
fasterMart = FasterMorphology2()
fasterEnd = FasterMorphology2()
vctWP = fasterWP << textWP
fasterMart += fasterWP
vctMart = fasterMart << textMart
fasterEnd += fasterWP
fasterEnd += fasterMart
vctEnd = fasterEnd << textEnd

In [119]:
print("Cosine similarity of War and Peace and Ender's Game", cosineSimilarity(vctWP, vctEnd))
print("Cosine similarity of Martian and Ender's Game", cosineSimilarity(vctMart, vctEnd))
print("Cosine similarity of War and Peace and Martian", cosineSimilarity(vctMart, vctWP))

print("Jaccard similarity of War and Peace and Ender's Game", JaccardCoefficient(vctWP, vctEnd))
print("Jaccard similarity of Martian and Ender's Game", JaccardCoefficient(vctMart, vctEnd))
print("Jaccard similarity of War and Peace and Martian", JaccardCoefficient(vctMart, vctWP))

dict function
Cosine similarity of War and Peace and Ender's Game 0.8844753562288444
dict function
Cosine similarity of Martian and Ender's Game 0.8553247137611864
dict function
Cosine similarity of War and Peace and Martian 0.8039365043485874
Jaccard similarity of War and Peace and Ender's Game 0.24310920362422508
Jaccard similarity of Martian and Ender's Game 0.32660272589601214
Jaccard similarity of War and Peace and Martian 0.20965073529411765


Да, при таком маленьком пересечении лексики слишком большое совпадение косинусных мер. Давайте посчитаем только для значимых частей речи.

In [120]:
# Все декораторы вынесены из определения класса, так как не дело это класса считать косинусную меру. Она сама по себе.
# А еще я заменил действие оператора << , теперь он возвращает словарь. Так все-таки проще считать косинусную меру.
# Заведен список значимых частей речи, в соответствии с которым проводится фильтрация результатов.
# Теперь у нас есть два закешированных списка: интересующей нас части речи и остальные.
# Как следствие, пришлось переписать все части, которые касаются кеширования результатов.
class FasterMorphology2:
    """ Класс для быстрого морфологического анализа текстов и их векторизации.
    """

    def __init__(self):  # Функция инициализации объекта после его создания.
        self.morpho = pymorphy2.MorphAnalyzer()
        # Теперь надо различать слова, которые нам нравятся, не нравятся и которые не встретились.
        self.cashPos = {}
        self.cashNeg = []
        # Добавим словарь для запоминания, на каком месте вектора находится какая начальная форма.
        self.dictionary = {}
        self.imPoS = ['ADJF', 'NOUN', 'VERB', 'INFN', 'PRTF', 'GRND']

    def analyzeWords(self, words):
        """ Проводит морфологический анализ списка токенов words.
            Возвращает список начальных форм слов.
        """
        res = []
        for word in words:
            if word in self.cashPos:  # Сперва ищем очередное слово в кеше.
                res.append(self.cashPos[word])
            if word in self.cashNeg:  # Сперва ищем очередное слово в кеше.
                pass
            else:  # Если его там нет, проводим морфологический анализ и кешируем.
                r = self.morpho.parse(word)
                if r[0].tag.POS in self.imPoS:
                    r = r[0].normal_form
                    res.append(r)
                    self.cashPos[word] = r
                    # Также для каждой начальной формы запоминаем ее позицию в векторе.
                    if r not in self.dictionary:
                        self.dictionary[r] = len(self.dictionary)
                else:
                    self.cashNeg.append(word)
        return res

    def breakByWords(self, text):
        """ Разбивает текст на русские слова.
        """
        return [w[0].lower() for w in re.findall("([А-ЯЁа-яё]+(-[А-ЯЁа-яё]+)*)", text)]

    def analyzeText(self, text):
        """ Проводит морфологический анализ строки с текстом text. 
            Выделяет из нее слова, написанные русской кириллицей.
            Возвращает список начальных форм слов.
        """
        words = self.breakByWords(text)
        return self.analyzeWords(words)

    # Вообще-то тоже самое умеет Counter, но ему надо сперва привести слова к начальной форме.
    def vectorizeAsDict(self, words):
        """ Возвращает векторное разреженное представление текста в виде словаря.
            Текст передается как список токенов words.
            Вместо позиции для индексации используется само слово.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words, str):
            words = self.breakByWords(words)

        vct = {}
        for word in words:  # Для каждого слова прповодим анализ.
            if word in self.cashPos:  # Это закешированное слово со значимой частью речи.
                # Считаем частоты слов.
                vct[self.cashPos[word]] = vct.get(self.cashPos[word], 0)+1
            elif word in self.cashNeg:  # Это закешированное слово не со значимой частью речи.
                pass
            else:
                r = self.morpho.parse(word)
                if r[0].tag.POS in self.imPoS:
                    r = r[0].normal_form
                    self.cashPos[word] = r
                    vct[r] = 1
                    if r not in self.dictionary:
                        self.dictionary[r] = len(self.dictionary)
                else:
                    # Мы ничего не хотим знать про это слово.
                    self.cashNeg.append(word)

        return vct

    def clearDict(self):
        """ Очищает словарь. Вдруг надо пересчитать так как изменилась размерность пространства.
        """
        self.dictionary = {}

    def formDict(self, texts):
        """ Сформировать словарь по тексту не формируя разметку текста.
        """
        for text in texts:
            for word in text:
                if word not in self.cashPos and word not in self.cashNeg:
                    r = self.morpho.parse(word)
                    if r[0].tag.POS in self.imPoS:
                        r = r[0].normal_form
                        self.cashPos[word] = r
                        if r not in self.dictionary:
                            self.dictionary[r] = len(self.dictionary)
                    else:
                        # Мы ничего не хотим знать про это слово.
                        self.cashNeg.append(word)

    def vectorizeAsList(self, words2):
        """ Возвращает векторное представление текста в виде плотного списка (включает нули).
            Текст передается как список токенов words.
            Позиция каждого слова в векторе определяется числом, хранимым в dictionary.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words2, str):
            words = self.breakByWords(words2)
        else:
            words = words2

        # Сперва обновляем dictionary.
        for word in words:
            if word in self.cashNeg:
                pass
            elif word not in self.cashPos:
                r = self.morpho.parse(word)
                if r[0].tag.POS in self.imPoS:
                    r = r[0].normal_form
                    self.cashPos[word] = r
                    if r not in self.dictionary:
                        self.dictionary[r] = len(self.dictionary.keys())
                else:
                    # Мы ничего не хотим знать про это слово.
                    self.cashNeg.append(word)
        # Теперь, когда все слова есть в кеше и словаре и известен размер вектора, можно приступать к векторизации.
        vct = [0 for _ in self.dictionary]
        for word in words:
            if word in self.cashPos:
                vct[self.dictionary[self.cashPos[word]]] += 1
        return vct

    def vectorizeAsList2(self, words):
        """ Возвращает векторное представление текста в виде плотного списка (включает нули).
            Текст передается как список токенов words. В вектор включаются только слова, находящиес в словаре.
            Позиция каждого слова в векторе определяется числом, хранимым в dictionary.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        vct = [0 for _ in self.dictionary]
        for word in words:
            if word in self.cashPos:
                vct[self.dictionary[self.cashPos[word]]] += 1
        return vct

    def vectorizeAsArray(self, words):
        """ Возвращает векторное представление текста в виде плотного массива (включает нули).
            Текст передается как список токенов words.
            Позиция каждого слова в векторе определяется числом, хранимым в dictionary.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words, str):
            words = self.breakByWords(words)

        # Сперва обновляем dictionary.
        for word in words:
            if word in self.cashNeg:
                pass
            elif word not in self.cashPos:
                r = self.morpho.parse(word)
                if r[0].tag.POS in self.imPoS:
                    r = r[0].normal_form
                    self.cashPos[word] = r
                    if r not in self.dictionary:
                        self.dictionary[r] = len(self.dictionary.keys())
                else:
                    # Мы ничего не хотим знать про это слово.
                    self.cashNeg.append(word)
        # Теперь, когда все слова есть в кеше и словаре и известен размер вектора, можно приступать к векторизации.
        vct = np.zeros((len(self.dictionary)))
        for word in words:
            if word in self.cashPos:
                vct[self.dictionary[self.cashPos[word]]] += 1
        return vct

    def __iadd__(self, other):
        """ Оператор добавления словаря от другого объекта.
        """
        self.cashPos.update(other.cashPos)
        self.cashNeg = list(set(self.cashNeg) | set(other.cashNeg))
        for word in set(other.dictionary.keys()) - set(self.dictionary.keys()):
            self.dictionary[word] = len(self.dictionary)
        return self

    def __lshift__(self, text):
        """ Оператор возвращает векторное представление текста в виде словаря.
        """
        return self.vectorizeAsDict(text)

    def __repr__(self):
        """ Текстовое представление объекта.
        """
        return "Object of class FasterMorphology2 <" + str(id(self)) + \
            ">\nPositive cash size: " + str(len(self.cashPos)) + \
            ">\nNegative cash size: " + str(len(self.cashNeg)) + \
            "\nDictionary size: " + str(len(self.dictionary))

    def __getitem__(self, key):
        """ Возвращает элемент словаря при помощи квадратных скобок.
        """
        if isinstance(key, slice):
            return list(self.dictionary.keys())[key.start: key.stop: key.step]
        else:
            return list(self.dictionary.keys())[key]

    def __bool__(self):
        """ Класс ведет себя как булевская переменная. Проверяет было ли что-нибудь закешировано.
        """
        return len(self.dictionary) != 0

    def __call__(self):
        """ Объект класса можно "вызвать" как функцию. Можно просто переопределить оператор "круглые скобки".
        """
        print("-=* Overall results for FasterMorphology2*=-\nPositive cash size: " + str(len(self.cashPos)) +
              "Negative cash size: " + str(len(self.cashNeg)) +
              "\nDictionary size: " + str(len(self.dictionary)))
        return self


In [121]:
fasterWP = FasterMorphology2()
fasterMart = FasterMorphology2()
fasterEnd = FasterMorphology2()
vctWP = fasterWP << textWP
fasterMart += fasterWP
vctMart = fasterMart << textMart
fasterEnd += fasterWP
fasterEnd += fasterMart
vctEnd = fasterEnd << textEnd

In [122]:
"""
Old version
Cosine similarity of War and Peace and Ender's Game 0.8877558800566507
Cosine similarity of Martian and Ender's Game 0.8558887385483673
Cosine similarity of War and Peace and Martian 0.8042099465818117
Jaccard similarity of War and Peace and Ender's Game 0.24524534043362495
Jaccard similarity of Martian and Ender's Game 0.3274018379281537
Jaccard similarity of War and Peace and Martian 0.2114193992203623
"""

print("Cosine similarity of War and Peace and Ender's Game", cosineSimilarity(vctWP, vctEnd))
print("Cosine similarity of Martian and Ender's Game", cosineSimilarity(vctMart, vctEnd))
print("Cosine similarity of War and Peace and Martian", cosineSimilarity(vctMart, vctWP))

print("Jaccard similarity of War and Peace and Ender's Game", JaccardCoefficient(vctWP, vctEnd))
print("Jaccard similarity of Martian and Ender's Game", JaccardCoefficient(vctMart, vctEnd))
print("Jaccard similarity of War and Peace and Martian", JaccardCoefficient(vctMart, vctWP))


dict function
Cosine similarity of War and Peace and Ender's Game 0.5732908932996438
dict function
Cosine similarity of Martian and Ender's Game 0.6641460780523735
dict function
Cosine similarity of War and Peace and Martian 0.6179231795579522
Jaccard similarity of War and Peace and Ender's Game 0.23238840702348212
Jaccard similarity of Martian and Ender's Game 0.3082328363878074
Jaccard similarity of War and Peace and Martian 0.19565994383456728


При небольшом уменьшении меры Жаккара косинусная мера значительно уменьшилась. Помимо этого, поменялась и схожесть текстов: теперь "Марсианин" больше похож на "Игру Эндера", чем на "Войну и мир".

Представим себе теперь, что мы хотим получить доступ к длине словаря, но при этом хотим организовать доступ так, чтобы пользователь не обращался к самому словарю. Для этого у нас есть два пути: написать функцию, которая будет возвращать длину словаря, или использовать `@property`.

In [123]:
class PartOfMorpho1:

    def __init__(self):
        self.morpho = pymorphy2.MorphAnalyzer()
        # Теперь надо различать слова, которые нам нравятся, не нравятся и которые не встретились.
        self.cashPos = {}
        self.cashNeg = []
        # Добавим словарь для запоминания, на каком месте вектора находится какая начальная форма.
        self.dictionary = {}
        self.imPoS = ['ADJF', 'NOUN', 'VERB', 'INFN', 'PRTF', 'GRND']

    def clear_dict(self):
        """ Очищает словарь. Вдруг надо пересчитать так как изменилась размерность пространства.
        """
        self.dictionary = {}

    def breakByWords(self, text):
        """ Разбивает текст на русские слова.
        """
        return [w[0].lower() for w in re.findall("([А-ЯЁа-яё]+(-[А-ЯЁа-яё]+)*)", text)]

    def vectorizeAsDict(self, words):
        """ Возвращает векторное разреженное представление текста в виде словаря.
            Текст передается как список токенов words.
            Вместо позиции для индексации используется само слово.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words, str):
            words = self.breakByWords(words)

        vct = {}
        for word in words:  # Для каждого слова прповодим анализ.
            if word in self.cashPos:  # Это закешированное слово со значимой частью речи.
                # Считаем частоты слов.
                vct[self.cashPos[word]] = vct.get(self.cashPos[word], 0)+1
            elif word in self.cashNeg:  # Это закешированное слово не со значимой частью речи.
                pass
            else:
                r = self.morpho.parse(word)
                if r[0].tag.POS in self.imPoS:
                    r = r[0].normal_form
                    self.cashPos[word] = r
                    vct[r] = 1
                    if r not in self.dictionary:
                        self.dictionary[r] = len(self.dictionary)
                else:
                    # Мы ничего не хотим знать про это слово.
                    self.cashNeg.append(word)

        return vct

    def get_dict_len(self):
        """ Решение при помощи специальной функции.
        """
        return len(self.dictionary.keys())


In [124]:
mo1 = PartOfMorpho1()
mo1.vectorizeAsDict(textEnd)
mo1.get_dict_len()

6831

In [29]:
class PartOfMorpho2:

    def __init__(self):
        self.morpho = pymorphy2.MorphAnalyzer()
        # Теперь надо различать слова, которые нам нравятся, не нравятся и которые не встретились.
        self.cashPos = {}
        self.cashNeg = []
        # Добавим словарь для запоминания, на каком месте вектора находится какая начальная форма.
        self.dictionary = {}
        self.imPoS = ['ADJF', 'NOUN', 'VERB', 'INFN', 'PRTF', 'GRND']

    def clear_dict(self):
        """ Очищает словарь. Вдруг надо пересчитать так как изменилась размерность пространства.
        """
        self.dictionary = {}

    def breakByWords(self, text):
        """ Разбивает текст на русские слова.
        """
        return [w[0].lower() for w in re.findall("([А-ЯЁа-яё]+(-[А-ЯЁа-яё]+)*)", text)]

    def vectorizeAsDict(self, words):
        """ Возвращает векторное разреженное представление текста в виде словаря.
            Текст передается как список токенов words.
            Вместо позиции для индексации используется само слово.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words, str):
            words = self.breakByWords(words)

        vct = {}
        for word in words:  # Для каждого слова прповодим анализ.
            if word in self.cashPos:  # Это закешированное слово со значимой частью речи.
                # Считаем частоты слов.
                vct[self.cashPos[word]] = vct.get(self.cashPos[word], 0)+1
            elif word in self.cashNeg:  # Это закешированное слово не со значимой частью речи.
                pass
            else:
                r = self.morpho.parse(word)
                if r[0].tag.POS in self.imPoS:
                    r = r[0].normal_form
                    self.cashPos[word] = r
                    vct[r] = 1
                    if r not in self.dictionary:
                        self.dictionary[r] = len(self.dictionary)
                else:
                    # Мы ничего не хотим знать про это слово.
                    self.cashNeg.append(word)

        return vct

    @property
    def get_dict_len(self):
        """ Решение при помощи декоратора @property.
        """
        return len(self.dictionary.keys())


In [30]:
mo2 = PartOfMorpho2()
mo2.vectorizeAsDict(textEnd)
mo2.get_dict_len

NameError: name 'pymorphy2' is not defined

Обратите внимание, что писать в такое свойство нельзя. Это позволяет ограничить доступ на чтение и на запись.

In [127]:
mo2.get_dict_len = 0

AttributeError: can't set attribute 'get_dict_len'

Но представим себе, что такое в принципе возможно. Например, мы можем оставить в словаре только N самых частотных слов, если размер словаря больше N. В таком случае опять возможны два решения - с помощью специальной функции и декоратора. Рассмотрим только второй вариант - от него просто перейти к первому.

In [128]:
class PartOfMorpho3:

    def __init__(self):
        self.morpho = pymorphy2.MorphAnalyzer()
        # Теперь надо различать слова, которые нам нравятся, не нравятся и которые не встретились.
        self.cashPos = {}
        self.cashNeg = []
        # Добавим словарь для запоминания, на каком месте вектора находится какая начальная форма.
        self.dictionary = {}
        self.imPoS = ['ADJF', 'NOUN', 'VERB', 'INFN', 'PRTF', 'GRND']

    def clear_dict(self):
        """ Очищает словарь. Вдруг надо пересчитать так как изменилась размерность пространства.
        """
        self.dictionary = {}

    def breakByWords(self, text):
        """ Разбивает текст на русские слова.
        """
        return [w[0].lower() for w in re.findall("([А-ЯЁа-яё]+(-[А-ЯЁа-яё]+)*)", text)]

    def vectorizeAsDict(self, words):
        """ Возвращает векторное разреженное представление текста в виде словаря.
            Текст передается как список токенов words.
            Вместо позиции для индексации используется само слово.
            Возвращает словарь с начальными формами в ключах и частотами этих форм.
        """
        # Если это был текст - разбиваем на слова.
        if isinstance(words, str):
            words = self.breakByWords(words)

        vct = {}
        for word in words:  # Для каждого слова прповодим анализ.
            if word in self.cashPos:  # Это закешированное слово со значимой частью речи.
                # Считаем частоты слов.
                vct[self.cashPos[word]] = vct.get(self.cashPos[word], 0)+1
            elif word in self.cashNeg:  # Это закешированное слово не со значимой частью речи.
                pass
            else:
                r = self.morpho.parse(word)
                if r[0].tag.POS in self.imPoS:
                    r = r[0].normal_form
                    self.cashPos[word] = r
                    vct[r] = 1
                    if r not in self.dictionary:
                        self.dictionary[r] = len(self.dictionary)
                else:
                    # Мы ничего не хотим знать про это слово.
                    self.cashNeg.append(word)

        return vct

    @property
    def dict_len(self):
        """ Решение при помощи декоратора @property.
        """
        return len(self.dictionary.keys())

    @dict_len.setter
    def dict_len(self, length):
        """ Решение при помощи декоратора @property.
        """
        if length < len(self.dictionary.keys()):
            self.dictionary = {x[0]: x[1] for x in
                               sorted(self.dictionary.items(), key=lambda x: x[1])[:length]}


In [129]:
mo3 = PartOfMorpho3()
mo3.vectorizeAsDict(textEnd)
print(mo3.dict_len)
mo3.dict_len = 100
mo3.dictionary.items()

6831


dict_items([('орсон', 0), ('карда', 1), ('скотт', 2), ('игра', 3), ('эндёр', 4), ('один', 5), ('самый', 6), ('яркий', 7), ('имя', 8), ('современный', 9), ('научный', 10), ('фантастика', 11), ('произведение', 12), ('писатель', 13), ('высокий', 14), ('премия', 15), ('хьюго', 16), ('небьюла', 17), ('локус', 18), ('главное', 19), ('мера', 20), ('талант', 21), ('выделять', 22), ('творение', 23), ('хороший', 24), ('космический', 25), ('роман', 26), ('выступать', 27), ('неподдельный', 28), ('оригинальность', 29), ('сюжет', 30), ('откровенный', 31), ('мощный', 32), ('эмоциональность', 33), ('история', 34), ('эндрю', 35), ('уиггин', 36), ('великий', 37), ('полководец', 38), ('эра', 39), ('межзвёздный', 40), ('флот', 41), ('земля', 42), ('вести', 43), ('отчаянный', 44), ('борьба', 45), ('жестокий', 46), ('негуманоидный', 47), ('пришелец', 48), ('отобрать', 49), ('ребёнок', 50), ('военный', 51), ('готовить', 52), ('особый', 53), ('программа', 54), ('командный', 55), ('состав', 56), ('земной', 57)

Есть, правда, один нюанс. Мы забыли посчитать частоты слов в тексте, когда формировали словарь. Так что придется обойтись просто первой сотней слов текста.  
Но сеттеры и геттеры от этого работать не перестали и отражают идею своего использования.

## Исключительные ситуации

В Питоне есть возможность отлавливать происходящие ошибки при помощи try ... except. Если в блоке try происходит ошибка, то выполнение программы не прекращается, а выполняется сперва блок except, а потом код выполняется дальше.

In [2]:
while True:
    try:
        d1 = input("введите число ")
        d2 = 10 / int(d1)
        print(d2)
        break
    except:
        print("Введите другое число!")

введите число  0


Введите другое число!


введите число  0


Введите другое число!


введите число  1


10.0


In [3]:
try:
    a = b / 0
except:
    print("Exception comes!")

Exception comes!


In [4]:
# Без исключительной ситуации.4
a = b / 0


Замечу, что некоторые функции специально вызывают исключительные ситуации, если в них что-то пошло не по умолчанию. Это может быть штатным поведением функции. 

Если хочется поймать ситуацию, когда код выполнился без исключительной ситуации, нужно использовать конструкцию `try ... except ... else ...`. Блок `else` выполняется только в том случае, когда код выполнился штатно, то есть полностью и корректно. В такой ситуации можно ожидать корректного состояния программы по всем пунктам.

In [7]:
try:
    f = open("no_file_at_disk.none")
except:
    data = None
else:
    f.seek(0, 2) # move the cursor to the end of the file
    size = f.tell()
    data = f.read(size)
    
print(data[:100])

TypeError: 'NoneType' object is not subscriptable

Иногда бывает необходимо выполнить код после в любом случае - произошло исключение или нет. Например, нам надо освободить какие-то ресурсы. Для этого можно использовать блок `finally`. Если исключения не будет, он выполнится и код продолжит свое выполнение. Если исключение произойдет для `try ... finally ... `, выполнится блок `finally`, а только потом произойдет исключение. Если оно произойдет в конструкции `try ... except ... finally ...`, сперва отработает код исключения, а потом `finally`.

`finally` идет после `else`.

In [5]:
# b = None
b = 1
c = 0
# c = 1
try:
    a = b / c
# except:
#     print("Exception comes!")
# else:
#     print("Try not to get worried, try not to turn on to Problems that upset you, oh.")
finally:
    print("I'll be back. 👍")


I'll be back. 👍


ZeroDivisionError: division by zero

Мы можем поймать **переменную** исключения и посмотреть на нее.

In [6]:
# b = None
b = 1
c = 0
# c = 1

try:
    a = b / c
except Exception as err:
    print(type(err), dir(err), dir(err.__traceback__))#, err.with_traceback())


<class 'ZeroDivisionError'> ['__cause__', '__class__', '__context__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__suppress_context__', '__traceback__', 'args', 'with_traceback'] ['tb_frame', 'tb_lasti', 'tb_lineno', 'tb_next']


Как можно увидеть из предыдущего примера, исключения бывают разные, мы можем сказать какие именно ловить.

In [7]:
# b = None
b = 1
c = 0
# c = 1

try:
    a = b / c
except TypeError as err:
    print('Take care on your variables', err.args)
except ZeroDivisionError as err:
    print('Never divide by zero', err.args)
except Exception as err:
    print(type(err), dir(err))


Never divide by zero ('division by zero',)


Можно порождать собственные исключения. Для этого используем `raise`.

In [8]:
b'\xfe\xff'.decode('utf-8')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xfe in position 0: invalid start byte

In [9]:
def never_work() -> None:
    raise
    
def never_work2() -> None:
    raise Exception("I'm booooreeeed!")
    
def never_work3() -> None:
    err = ZeroDivisionError("I'm looking like zero division!", {'a':1, 'b':2})
    err.it_wents_wrong = ['asd', 12]
    raise err    
    

In [10]:
never_work()

RuntimeError: No active exception to reraise

In [ ]:
never_work2()

Exception: I'm booooreeeed!

In [15]:
try:
    never_work3()
except ZeroDivisionError as err:
    print('what?', err.args, err.it_wents_wrong)

what? ("I'm looking like zero division!", {'a': 1, 'b': 2}) ['asd', 12]


### namedtuple

Есть старая проблема: мне не очень понятно, почему я храню объекты в словарях, списках или кортежах. С другой стороны, я хочу, чтобы значения полей объекта никто не менял. Или чтобы оно имело только простые свойства практически без методов. Или создавать классы и объекты налету. 

Для этого есть старое решение - `namedtuple`. Ей можно передать список названий, по которым мы будем обращаться к элементам кортежа. При этом названия можно использовать как свойства, то есть писать после имени объекта через точку. 

In [5]:
from collections import namedtuple

In [22]:
# Создаем новый тип, передаем в него название типа и список полей.
LentaArticleTupled = namedtuple('LentaArticleTupled', 
                                ['title', 'text', 'description', 'time', 'author'])


In [24]:
# Создаем объект этого нового типа, передаем в него значения полей.
la = LentaArticleTupled('123', '234', 'asdf', '10:01', 'Nope')
la2 = LentaArticleTupled('321', text='098', description='asdf', time='10:01', author='Nope')

In [25]:
la.title

'123'

Кстати, это всё ещё кортеж, то есть можно обращаться к полям по номеру.

In [26]:
la[1]

'234'

Менять значения полей не получится - это же кортеж!

In [27]:
la.title = '123'

AttributeError: can't set attribute

А что внутри?

In [28]:
type(LentaArticleTupled)

type

In [29]:
dir(LentaArticleTupled)

['__add__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getnewargs__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '_asdict',
 '_field_defaults',
 '_fields',
 '_make',
 '_replace',
 'author',
 'count',
 'description',
 'index',
 'text',
 'time',
 'title']

In [30]:
LentaArticleTupled.title

_tuplegetter(0, 'Alias for field number 0')

Создадим теперь функцию, которая загружает новость с Ленты.ру и возвращает именованный кортеж.

In [31]:
def get_lenta_article_tupled(url: str) -> LentaArticleTupled:
    page = requests.get(url)
    tree = html.fromstring(page.text)
    article = LentaArticleTupled(
            tree.xpath(".//h1")[0].text_content(),
            tree.xpath(".//div[contains(@class, 'topic-authors')]")[0].text_content().strip(),
            tree.xpath(".//meta[@property='og:description']")[0].get("content"),
            tree.xpath(".//time[contains(@class, 'topic-header__time')]")[0].text_content().strip(), 
            '\n'.join([p.text_content() for p in 
                tree.xpath(".//div[contains(@class, '_news')]//p[contains(@class, 'topic-body__content-text')]")]
                    )
           )
    
    return article

get_lenta_article_tupled('https://lenta.ru/news/2021/02/27/apple_effect/')

IndexError: list index out of range

### @dataclass

Иногда нам необходимо создать класс, который будет содержать в себе только данные, но не будет содержать в себе методов работы с этими данными. Для этого существует декоратор `dataclass` из библиотеки `dataclasses`.

Их удобство заключается в том, что можно просто описать поля, входящие в этот класса, и все объекты будут создаваться с этими полями. При этом обязательно надо указывать тип атрибута. (Но можно указать `Any` из модуля `typing`.)

При необходимости можно присвоить атрибутам значения по умолчанию. Но следует иметь в виду, что сперва идут все поля без значений по умолчанию, а потом все с присваиваемыми значениями.

Заметим, что в таких классах могут быть и методы. Просто иногда проще описать такой класс и ничего в него не добавлять, а значения по умолчанию пусть берутся из описания.

Более подробно про них можно посмотреть [здесь](https://habr.com/ru/post/415829/) и [здесь](https://docs.python.org/3/library/dataclasses.html)


In [3]:
from dataclasses import dataclass

In [35]:
@dataclass
class LentaArticle:
    title: str
    text: str
    description: str
    time: str = "00:00"
    author: str = "No author"
        


Теперь напишем функцию, которая будет возвращать объект новости, а не будет хранить атрибуты одной сущности в разных местах.

In [56]:
def get_lenta_article(url: str) -> LentaArticle:
    page = requests.get(url)
    tree = html.fromstring(page.text)
    ttl = tree.xpath(".//h1")[0].text_content()
    dscrptn = tree.xpath(".//meta[@property='og:description']")[0].get("content")

    txt = '\n'.join([p.text_content() for p in 
             tree.xpath(".//div[contains(@class, '_news')]//p[contains(@class, 'topic-body__content-text')]")]
                    )
    
    article = LentaArticle(ttl, txt, dscrptn)
    article.time = tree.xpath(".//time[contains(@class, 'topic-header__time')]")[0].text_content().strip()
    article.author = tree.xpath(".//div[contains(@class, 'topic-authors')]")[0].text_content().strip()
    return article

get_lenta_article('https://lenta.ru/news/2021/02/27/apple_effect/')

IndexError: list index out of range

Вообще-то, как-то не очень красиво. А давайте, раз уж можно заводить функции, создадим конструктор, в который будут передаваться значения в удобном для нас порядке. А заодно заведем метод `__repr__`.

In [36]:
@dataclass()
class LentaArticle:
    title: str
    text: str
    description: str
    time: str = "00:00"
    author: str = "No author"
        
    def __init__(self: 'LentaArticle', _title: str, _author: str, _description: str,
                 _time: str, _text: str):
        self.title = _title
        self.author = _author
        self.description = _description
        self.time = _time
        self.text = _text
        
    def __repr__(self: 'LentaArticle') -> str:
        return f"""LentaArticle(\n  title={self.title[:60]}\n  author={self.author}\n  """\
               f"""time={self.time}\n  {self.text[:100]}..."""
        
        
def get_lenta_article(url: str) -> LentaArticle:
    page = requests.get(url)
    tree = html.fromstring(page.text)
    article = LentaArticle(
            tree.xpath(".//h1")[0].text_content(),
            tree.xpath(".//div[contains(@class, 'topic-authors')]")[0].text_content().strip(),
            tree.xpath(".//meta[@property='og:description']")[0].get("content"),
            tree.xpath(".//time[contains(@class, 'topic-header__time')]")[0].text_content().strip(), 
            '\n'.join([p.text_content() for p in 
                tree.xpath(".//div[contains(@class, '_news')]//p[contains(@class, 'topic-body__content-text')]")]
                    )
           )
    
    return article

get_lenta_article('https://lenta.ru/news/2021/02/27/apple_effect/')

IndexError: list index out of range

Декоратор `@dataclass` обладает целым рядом параметров, помогающих проще решать некоторые задачи. 

`frozen=True` - в объекты нельзя будет добавлять новые атрибуты.  
`init, repr, eq, order =True` - заводят соответствующие функции по умолчанию: конструктор, представления, эквивалентности, сравнения.

In [57]:
@dataclass(frozen=True)
class LentaArticleFrozen:
    title: str = ""
    text: str = ""
    description: str = ""
    time: str = "00:00"
    author: str = "No author"
        
aaa = LentaArticleFrozen()
aaa.newOne = 1

FrozenInstanceError: cannot assign to field 'newOne'

Запустим вот такой код. И что мы увидим?

In [62]:
@dataclass()
class LentaArticleMAuthors:
    title: str
    text: str
    description: str
    time: str = "00:00"
    author: str = []

ValueError: mutable default <class 'list'> for field author is not allowed: use default_factory

1. Никто не следит за типами значений.
2. Присваивать мутабельные типы нельзя. Предлагают использовать `default_factory`.

In [37]:
from dataclasses import field

@dataclass()
class LentaArticleMAuthors:
    title: str
    text: str
    description: str
    time: str = "00:00"
    author: list[str] = field(default_factory=list)

In [38]:
class My2:
    ttt = []
    
@dataclass
class My3:
    tttt: My2 = My2
        
aaa = My3()
dir(aaa)

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'tttt']